In [1]:
import pandas as pd
import sqlalchemy as db

from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

from dotenv import load_dotenv
import os

load_dotenv("../configuration/.env")

True

# Data Warehouse Connection

In [2]:
# connect to MySQL database data warehouse
username =os.getenv('MYSQL_USER')
password = (':' + os.getenv('MYSQL_PASSWORD')) if os.getenv('MYSQL_PASSWORD') != None else ''
host = os.getenv('MYSQL_HOST')
port = os.getenv('MYSQL_PORT')
database = os.getenv('MYSQL_DATABASE_DW')

uriMysql = f"mysql://{username}{password}@{host}/{database}"
print(uriMysql)
clientMysql= db.create_engine(uriMysql)

mysql://root:@localhost/dw_netflix


# EXTRACT STAGE

## MYSQL OLTP 

In [3]:
# connect to MySQL OLTP
username =os.getenv('MYSQL_USER')
password = (':' + os.getenv('MYSQL_PASSWORD')) if os.getenv('MYSQL_PASSWORD') != None else ''
host = os.getenv('MYSQL_HOST')
port = os.getenv('MYSQL_PORT')
database = os.getenv('MYSQL_DATABASE_OLTP')

uri_mysql_OLTP = f"mysql://{username}{password}@{host}/{database}"
print(uri_mysql_OLTP)

connection_OLTP = db.create_engine(uri_mysql_OLTP)

mysql://root:@localhost/db_movies_netflix_transact


In [4]:
# SQL query to retrieve movie details along with their genre and participants (actors, directors, etc.)
query = """
SELECT
    movie.movieID as movieID, movie.movieTitle as title, movie.releaseDate as releaseDate,
    genre.name as genre, person.name as participantName, performer.performerRole as performerRole
FROM movie
INNER JOIN performer ON movie.movieID=performer.movieID
INNER JOIN person ON person.personID = performer.personID
INNER JOIN movie_genre ON movie.movieID = movie_genre.movieID
INNER JOIN genre ON movie_genre.genreID = genre.genreID
"""

# Execute the SQL query and load the result into a DataFrame
df_movies_oltp = pd.read_sql(query, con=connection_OLTP)

# Display the DataFrame with movie details
df_movies_oltp.head()

,movieID,title,releaseDate,genre,participantName,performerRole
0,M009,Braveheart,1995-05-24,Action,Tom Hanks,Actor
1,M009,Braveheart,1995-05-24,Action,Steven Spielberg,Director
2,M011,The Dark Knight,2008-07-18,Action,Johnny Depp,Actor
3,M011,The Dark Knight,2008-07-18,Action,Christopher Nolan,Director
4,M014,Avatar,2009-12-18,Action,Leonardo DiCaprio,Actor


## MongoDB

In [5]:
# conexion a mongo
username =os.getenv('MONGODB_USERNAME')
password = os.getenv('MONGODB_PASSWORD')
host = os.getenv('MONGODB_HOST')
database = os.getenv('MONGODB_DATABASE')

uri_Mongo = f"mongodb+srv://{username}:{password}@{host}={database}"
print(uri_mysql_OLTP)

clientMongo = MongoClient(uri_Mongo, server_api=ServerApi('1'))

mysql://root:@localhost/db_movies_netflix_transact


In [6]:
# Access the database
dbMongo = clientMongo['netflix_movies']

# Access the collections
movies = dbMongo['movies_df']
movies_documents = list(movies.find())
df_movies = pd.DataFrame(movies_documents)

interactions = dbMongo['user_netflix_interactions']
interactions_documents = list(interactions.find())
df_interactions = pd.DataFrame(interactions_documents)

df_movies.info()
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   _id                   9125 non-null   object 
 1   title                 9124 non-null   object 
 2   gender                9125 non-null   object 
 3   releaseDate           9125 non-null   object 
 4   AwardMovie            9125 non-null   object 
 5   netflix_score         7322 non-null   float64
 6   imdb_score            7309 non-null   float64
 7   sensacine_score       7360 non-null   float64
 8   rottentomatoes_score  7286 non-null   float64
dtypes: float64(4), object(5)
memory usage: 641.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9125 entries, 0 to 9124
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   _id                  9125 non-null   object 
 1   user              

# TRANSFORM STAGE

## preprocess interactions and movies collections

In [7]:
df_movies = df_movies.rename(columns={"gender":"genre"}).reset_index(names="movieID").drop(columns=["_id"])
df_movies["movieID"] = df_movies["movieID"].astype(str)
df_interactions = df_interactions.drop(columns=["_id"]).reset_index(names="userID")

print(df_movies.columns)
print("-------------")
print(df_interactions.columns)

Index(['movieID', 'title', 'genre', 'releaseDate', 'AwardMovie',
       'netflix_score', 'imdb_score', 'sensacine_score',
       'rottentomatoes_score'],
      dtype='object')
-------------
Index(['userID', 'user', 'movieTitle', 'finishCount', 'backClickCount',
       'movieForwardCount', 'playCount', 'movieViewPercentage'],
      dtype='object')


 movies dataframes normalization

 movieIndex==permormerID

In [8]:
# add index so as to identify permormerID with a movie through movieIndex (later movieIndex==permormerID)
df_movies_oltp.reset_index(names="movieIndex", inplace=True)


## get users dimention

In [9]:
df_dim_users = df_interactions[["userID","user"]].copy()
df_dim_users = df_dim_users.rename(columns={"user": "username"})
df_dim_users.head()

,userID,username
0,0,user_4
1,1,user_26
2,2,user_7
3,3,user_20
4,4,user_17


## get interactions dimention

"userID"=="interactionID"

In [10]:
df_dim_interactions = df_interactions.copy().rename(columns={"userID":"interactionID"}).drop(columns=["movieTitle","user"])
df_dim_interactions.head()

,interactionID,finishCount,backClickCount,movieForwardCount,playCount,movieViewPercentage
0,0,1,2,1,2,100.00
1,1,2,1,3,1,61.63
2,2,2,8,0,3,100.00
3,3,3,2,4,2,49.76
4,4,2,1,1,1,100.00


## movies dimention

In [11]:
df_dim_movies = df_movies[["movieID","title", "genre","releaseDate","AwardMovie"]].copy()
df_dim_movies = df_dim_movies.rename(columns={"AwardMovie":"award"})

df_dim_movies.head()

,movieID,title,genre,releaseDate,award
0,0,"Trip to the Moon, A (Voyage dans la lune, Le)",Action,1902-01-01,Grammy
1,1,"Birth of a Nation, The",Drama,1915-01-01,Sin Info
2,2,Intolerance: Love's Struggle Throughout the Ages,Drama,1916-01-01,Oscar
3,3,"20,000 Leagues Under the Sea",Action,1916-01-01,Oscar
4,4,"Immigrant, The",Comedy,1917-01-01,Sin Info


## performers dimention

In [12]:
df_dim_performers = df_movies_oltp[["movieIndex","participantName","performerRole"]].copy()
df_dim_performers.rename(columns={
  "movieIndex":"performerID",
  "participantName":"name",
  "performerRole":"role",
  }, inplace=True)

df_dim_performers.head()

,performerID,name,role
0,0,Tom Hanks,Actor
1,1,Steven Spielberg,Director
2,2,Johnny Depp,Actor
3,3,Christopher Nolan,Director
4,4,Leonardo DiCaprio,Actor


## score dimention

In [13]:
df_dim_score = df_movies[["title","netflix_score","imdb_score","sensacine_score","rottentomatoes_score"]].copy()
df_dim_score = df_dim_score.reset_index(names="scoreID")

# Check for duplicates on a pivot table where the index is the title and the value is the number of times a movie title is repeated
duplicates = df_movies.pivot_table(index=['title'], aggfunc='size')
duplicates = duplicates[duplicates > 1]

print("Example Duplicated Movie:")
print("\n")
print(df_dim_score[df_dim_score['title']==duplicates.sort_index().head(1).index[0]])

# merge duplicated values keeping the maximum score if two movies have the same title and different scores
df_dim_score = df_movies.groupby('title').agg({
    'netflix_score': 'max',
    'imdb_score': 'max',
    'sensacine_score': 'max',
    'rottentomatoes_score': 'max'
}).reset_index()

# Reset the index and assign unique IDs
df_dim_score.reset_index(drop=True, inplace=True)
df_dim_score['scoreID'] = df_dim_score.index

#print a duplicated movie and its scores
print("\n")
print("\nAggregated Movies with Maximum Scores:\n")
print(df_dim_score[df_dim_score['title']==duplicates.sort_index().head(1).index[0]])

# rename columns
df_dim_score.rename(columns={
    'netflix_score': 'netflixScore',
    'imdb_score': 'imdbScore',
    'sensacine_score': 'sensacineScore',
    'rottentomatoes_score': 'rottentomatoesScore'
},inplace=True)

Example Duplicated Movie:


      scoreID          title  netflix_score  imdb_score  sensacine_score  \
695       695  12 Angry Men             NaN         1.0              4.0   
4660     4660  12 Angry Men             2.0         4.0              2.0   

      rottentomatoes_score  
695                    3.0  
4660                   4.0  



Aggregated Movies with Maximum Scores:

            title  netflix_score  imdb_score  sensacine_score  \
30  12 Angry Men             2.0         4.0              4.0   

    rottentomatoes_score  scoreID  
30                   4.0       30  


# LOAD STAGE

In [14]:
# check last movie id to avoid duplicated primary key error
query_last_movie_id = """
SELECT MAX(movieID) FROM dimmovie;
"""

last_movie_id_prefix = pd.read_sql(query_last_movie_id, con=clientMysql).iloc[0,0]
df_dim_movies["movieID"] = last_movie_id_prefix + df_dim_movies["movieID"].astype(str)

# Load dimMovie data
df_dim_movies.to_sql('dimmovie', con=clientMysql, if_exists='append', index=False,index_label='movieID')

9125

In [15]:
query_last_user_id = """
SELECT MAX(userID) FROM dimuser;
"""

last_user_id = int(pd.read_sql(query_last_user_id, con=clientMysql).iloc[0, 0])
df_dim_users["userID"] = df_dim_users["userID"].astype(int)
df_dim_users["userID"] = df_dim_users["userID"] + last_user_id + 1

# Load dimUser data
df_dim_users.to_sql('dimuser', con=clientMysql, if_exists='append', index=False)

9125

In [16]:
query_last_interaction_id = """
SELECT MAX(interactionID) FROM diminteraction;
"""

last_interaction_id = int(pd.read_sql(query_last_interaction_id, con=clientMysql).iloc[0, 0])
df_dim_interactions["interactionID"] = df_dim_interactions["interactionID"].astype(int)
df_dim_interactions["interactionID"] = df_dim_interactions["interactionID"] + last_interaction_id + 1

# Load dimInteraction data
df_dim_interactions.to_sql('diminteraction', con=clientMysql, if_exists='append', index=False)

9125

In [17]:
query_last_score_id = """
SELECT MAX(scoreID) FROM dimscore;
"""

last_score_id = int(pd.read_sql(query_last_score_id, con=clientMysql).iloc[0, 0])
df_dim_score["scoreID"] = df_dim_score["scoreID"].astype(int)
df_dim_score["scoreID"] = df_dim_score["scoreID"] + last_score_id + 1

# Load dimScore data
df_dim_score[["scoreID","netflixScore","imdbScore","sensacineScore","rottentomatoesScore"]].to_sql('dimscore', con=clientMysql, if_exists='append', index=False)

8892

In [18]:
query_last_performer_id = """
SELECT MAX(performerID) FROM dimperformer;
"""

last_performer_id = int(pd.read_sql(query_last_performer_id, con=clientMysql).iloc[0, 0])
df_dim_performers["performerID"] = df_dim_performers["performerID"].astype(int)
df_dim_performers["performerID"] = df_dim_performers["performerID"] + last_performer_id + 1

# Load performers data
df_dim_performers.to_sql('dimperformer', con=clientMysql, if_exists='append', index=False)

58

## prepare fact watch

In [19]:
# Map interactions to their users
df_fact_watchs =df_interactions[["userID","movieTitle","user"]].copy()
df_fact_watchs.columns

Index(['userID', 'movieTitle', 'user'], dtype='object')

In [20]:
# Map users to their IDs
df_dim_user = pd.read_sql('SELECT userID,username FROM dimuser', con=clientMysql)
df_fact_watchs= df_fact_watchs.merge(df_dim_user, on="userID").drop(columns=['user'])

df_fact_watchs.columns

Index(['userID', 'movieTitle', 'username'], dtype='object')

In [21]:
# Map movies to their IDs
df_dim_movie= pd.read_sql('SELECT movieID,title FROM dimmovie', con=clientMysql)
df_fact_watchs = df_fact_watchs.merge(df_dim_movie, left_on='movieTitle', right_on='title').drop(columns=['title'])

df_fact_watchs.columns

Index(['userID', 'movieTitle', 'username', 'movieID'], dtype='object')

In [25]:
# Map performers to the movies they participate in
# keep in mind that we have defined movieIndex==permormerID before
df_dim_performers = df_movies_oltp[["movieIndex"]]
df_dim_performers.rename(columns={"movieIndex": "performerID"}, inplace=True)
df_dim_performers["performerID"] = df_dim_performers["performerID"].astype(int)
df_fact_watchs["movieID"] = df_fact_watchs["movieID"].astype(int)
df_fact_watchs = df_fact_watchs.merge(df_dim_performers, left_on="movieID", right_on="performerID")

df_fact_watchs.columns

C:\Users\joa_g\AppData\Local\Temp\ipykernel_3420\849459441.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dim_performers.rename(columns={"movieIndex": "performerID"}, inplace=True)
C:\Users\joa_g\AppData\Local\Temp\ipykernel_3420\849459441.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dim_performers["performerID"] = df_dim_performers["performerID"].astype(int)


Index(['userID', 'movieTitle', 'username', 'movieID', 'performerID'], dtype='object')

In [26]:
# Map scores to their movies
df_fact_watchs = df_fact_watchs.merge(df_dim_score, left_on='movieTitle', right_on='title').drop(columns=['title'])
df_fact_watchs.columns


Index(['userID', 'movieTitle', 'username', 'movieID', 'performerID',
       'netflixScore', 'imdbScore', 'sensacineScore', 'rottentomatoesScore',
       'scoreID'],
      dtype='object')

load fact watch

In [28]:
# Prepare FactWatchs DataFrame
df_fact_watchs = df_fact_watchs[['userID', 'movieID','performerID','scoreID']]
df_fact_watchs['interactionID'] = df_fact_watchs[['userID']]
df_fact_watchs['movieID'] = df_fact_watchs['movieID'].astype(str)
df_fact_watchs.reset_index(names='id', inplace=True)

# Load FactWatchs data
df_fact_watchs.to_sql('factwatchs', con=clientMysql, if_exists='append', index=False)

print("Data loaded successfully")

Data loaded successfully


C:\Users\joa_g\AppData\Local\Temp\ipykernel_3420\3796182262.py:8: UserWarning: The provided table name 'FactWatchs' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_fact_watchs.to_sql('FactWatchs', con=clientMysql, if_exists='append', index=False)
